# Predicting COCO Detector Failures Under COCO-O Cartoon Shift

This notebook evaluates whether OOD scores help predict failures for a COCO-trained object detector under a COCO-O cartoon shift. We use COCO as the in-distribution reference set and evaluate on COCO validation plus COCO-O cartoon images. Each row is an image-class pair, so the notebook asks whether the detector is likely to fail for a specific class in a specific image.

The goal is not to claim that OOD scores generally predict object-detection failure. This is a focused demo for this detector, dataset, and domain shift. Detector confidence is expected to be a strong baseline; we test whether global image OOD and object-region/chip OOD add complementary signal in this setting.

## What This Demonstrates

- Build image-class rows from detections and ground truth.
- Compute detector confidence, global OOD, and chip-level OOD features.
- Compare failure-risk models for COCO validation and COCO-O cartoon.
- Inspect the highest-risk image-class examples visually.

In [ ]:
import os
from pathlib import Path

DATASETS_ROOT = Path(os.environ.get('OODKIT_DATASETS', '/datasets'))
COCO_ROOT = Path(os.environ.get('OODKIT_COCO_ROOT', DATASETS_ROOT / 'coco'))
COCO_O_ROOT = Path(os.environ.get('OODKIT_COCO_O_ROOT', DATASETS_ROOT / 'coco_ood'))
OOD_DOMAIN = os.environ.get('OODKIT_COCO_O_DOMAIN', 'cartoon')

TRAIN_ANN = COCO_ROOT / 'coco_annotations' / 'instances_train2017.json'
VAL_ANN = COCO_ROOT / 'coco_annotations' / 'instances_val2017.json'
TRAIN_IMAGES = COCO_ROOT / 'coco_train'
VAL_IMAGES = COCO_ROOT / 'coco_val'

BACKBONE = 'dinov2-small'
DETECTOR_MODEL = 'fasterrcnn_resnet50_fpn_v2'
DETECTION_SCORE_THRESHOLD = float(os.environ.get('OODKIT_DETECTION_SCORE_THRESHOLD', '0.25'))
IOU_THRESHOLD = float(os.environ.get('OODKIT_IOU_THRESHOLD', '0.5'))
HEAD_EPOCHS = int(os.environ.get('OODKIT_HEAD_EPOCHS', '3'))
HEAD_LR = float(os.environ.get('OODKIT_HEAD_LR', '1e-3'))
VIM_PCT_VARIANCE = float(os.environ.get('OODKIT_VIM_PCT_VARIANCE', '0.95'))

MAX_TRAIN_IMAGES = int(os.environ.get('OODKIT_MAX_TRAIN_IMAGES', '30000'))
MAX_VAL_IMAGES = int(os.environ.get('OODKIT_MAX_VAL_IMAGES', '15000'))
MAX_OOD_IMAGES = int(os.environ.get('OODKIT_MAX_OOD_IMAGES', '15000'))

BATCH_SIZE = int(os.environ.get('OODKIT_BATCH_SIZE', '32'))
DETECTOR_BATCH_SIZE = int(os.environ.get('OODKIT_DETECTOR_BATCH_SIZE', '4'))
NUM_WORKERS = int(os.environ.get('OODKIT_NUM_WORKERS', '4'))
PIN_MEMORY = None
PERSISTENT_WORKERS = NUM_WORKERS > 0
HIGH_CONFIDENCE_THRESHOLD = float(os.environ.get('OODKIT_HIGH_CONFIDENCE_THRESHOLD', '0.75'))
SEED = 7

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from PIL import Image
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from torch.utils.data import Dataset

from oodkit.contrib.coco.category_table import CocoCategoryTable
from oodkit.contrib.coco.discovery import discover_coco_ood
from oodkit.data.chip_dataset import ChipDataset
from oodkit.detection import (
    attach_ood_features,
    detection_chips_from_table,
    evaluate_detection_tables,
    run_torchvision_detector,
)
from oodkit.detectors import ViM
from oodkit.embeddings.backbones import load_backbone
from oodkit.embeddings.embedder import Embedder
from oodkit.failure import calibration_bins, evaluate_failure_baselines

rng = np.random.default_rng(SEED)

In [ ]:
def coco_tables(
    ann_path,
    image_root,
    category_table,
    *,
    dataset_name,
    image_id_prefix,
    max_images=None,
    seed=0,
):
    with open(ann_path, encoding='utf-8') as f:
        data = json.load(f)

    images = list(data.get('images', []))
    if max_images is not None and len(images) > max_images:
        keep = set(np.random.default_rng(seed).choice(len(images), size=max_images, replace=False))
        images = [img for i, img in enumerate(images) if i in keep]

    raw_to_img = {int(img['id']): img for img in images}
    raw_to_public = {
        raw_id: f'{image_id_prefix}:{Path(img["file_name"]).stem}'
        for raw_id, img in raw_to_img.items()
    }
    image_rows = []
    for raw_id, img in raw_to_img.items():
        image_rows.append(
            {
                'image_id': raw_to_public[raw_id],
                'source_image_id': Path(img['file_name']).stem,
                'coco_image_id': raw_id,
                'file_name': img['file_name'],
                'image_path': str(Path(image_root) / img['file_name']),
                'image_width': int(img.get('width', 0)),
                'image_height': int(img.get('height', 0)),
                'dataset_name': dataset_name,
            }
        )

    gt_rows = []
    for ann in data.get('annotations', []):
        raw_image_id = int(ann['image_id'])
        if raw_image_id not in raw_to_img or ann.get('iscrowd', 0):
            continue
        x, y, w, h = [float(v) for v in ann['bbox']]
        if w <= 0 or h <= 0:
            continue
        img = raw_to_img[raw_image_id]
        class_id = category_table.category_id_to_idx[int(ann['category_id'])]
        gt_rows.append(
            {
                'image_id': raw_to_public[raw_image_id],
                'class_id': class_id,
                'class_name': category_table.idx_to_name[class_id],
                'bbox': [x, y, x + w, y + h],
                'gt_id': f'{raw_to_public[raw_image_id]}_gt_{ann["id"]}',
                'image_path': str(Path(image_root) / img['file_name']),
                'image_width': int(img.get('width', 0)),
                'image_height': int(img.get('height', 0)),
                'dataset_name': dataset_name,
            }
        )

    return pd.DataFrame(image_rows), pd.DataFrame(gt_rows)


class ImagePathDataset(Dataset):
    def __init__(self, frame, processor):
        self.frame = frame.reset_index(drop=True)
        self.processor = processor
        self.imgs = [(p, -1) for p in self.frame['image_path'].tolist()]

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        path = self.frame.iloc[index]['image_path']
        with Image.open(path) as img:
            img = img.convert('RGB')
            return self.processor(images=img, return_tensors='pt')['pixel_values'].squeeze(0)


def image_maps(image_frame):
    paths = dict(zip(image_frame['image_id'], image_frame['image_path']))
    sizes = dict(zip(image_frame['image_id'], zip(image_frame['image_width'], image_frame['image_height'])))
    return paths, sizes


def add_reference_zscores(frame, score_cols, *, reference_domain='COCO-val'):
    out = frame.copy()
    ref_mask = out['dataset_name'].eq(reference_domain)
    for col in score_cols:
        ref = pd.to_numeric(out.loc[ref_mask, col], errors='coerce')
        mean = ref.mean()
        std = ref.std(ddof=0)
        if not np.isfinite(std) or std == 0:
            std = 1.0
        out[f'{col}_z'] = (pd.to_numeric(out[col], errors='coerce') - mean) / std
    out['combined_ood_z'] = out[[f'{c}_z' for c in score_cols]].fillna(0.0).sum(axis=1)
    return out


def classification_metrics(y_true, y_prob):
    y = np.asarray(y_true, dtype=int)
    p = np.asarray(y_prob, dtype=float)
    if len(np.unique(y)) < 2:
        return {'AUROC': np.nan, 'AUPR': np.nan, 'Brier': brier_score_loss(y, p)}
    return {
        'AUROC': roc_auc_score(y, p),
        'AUPR': average_precision_score(y, p),
        'Brier': brier_score_loss(y, p),
    }


def prediction_metrics_by_domain(predictions, features):
    pred = predictions.merge(
        features[['image_id', 'class_id', 'dataset_name']],
        on=['image_id', 'class_id'],
        how='left',
    )
    rows = []
    for (model, dataset_name), group in pred.groupby(['model', 'dataset_name'], sort=True):
        metric = classification_metrics(group['target'], group['predicted_failure_probability'])
        rows.append({'model': model, 'dataset_name': dataset_name, 'n': len(group), **metric})
    return pd.DataFrame(rows)


def confidence_ood_failure_table(
    frame,
    *,
    confidence_col='mean_detection_confidence',
    ood_col='combined_ood_z',
    confidence_bins=(0.5, 0.7, 0.9, 1.01),
):
    rows = []
    work = frame[np.isfinite(frame[confidence_col]) & np.isfinite(frame[ood_col])].copy()
    for lo, hi in zip(confidence_bins[:-1], confidence_bins[1:]):
        group = work[(work[confidence_col] >= lo) & (work[confidence_col] < hi)]
        if len(group) < 4:
            continue
        low_cut = group[ood_col].quantile(0.33)
        high_cut = group[ood_col].quantile(0.67)
        low = group[group[ood_col] <= low_cut]
        high = group[group[ood_col] >= high_cut]
        rows.append(
            {
                'confidence_bin': f'{lo:0.1f}-{min(hi, 1.0):0.1f}',
                'n': len(group),
                'low_ood_n': len(low),
                'low_ood_failure_rate': low['has_failure'].mean(),
                'high_ood_n': len(high),
                'high_ood_failure_rate': high['has_failure'].mean(),
            }
        )
    return pd.DataFrame(rows)


def print_metrics(metrics):
    display(metrics.sort_values('AUROC', ascending=False).reset_index(drop=True))

In [ ]:
category_table = CocoCategoryTable.from_coco_json(TRAIN_ANN)
class_names = dict(enumerate(category_table.names()))
ood_paths = discover_coco_ood(COCO_O_ROOT, only=[OOD_DOMAIN])[0]

train_images, train_gt = coco_tables(
    TRAIN_ANN,
    TRAIN_IMAGES,
    category_table,
    dataset_name='COCO-train',
    image_id_prefix='train',
    max_images=MAX_TRAIN_IMAGES,
    seed=SEED,
)
val_images, val_gt = coco_tables(
    VAL_ANN,
    VAL_IMAGES,
    category_table,
    dataset_name='COCO-val',
    image_id_prefix='id',
    max_images=MAX_VAL_IMAGES,
    seed=SEED + 1,
)
ood_images, ood_gt = coco_tables(
    ood_paths.ann,
    ood_paths.images,
    category_table,
    dataset_name=f'COCO-O-{OOD_DOMAIN}',
    image_id_prefix=OOD_DOMAIN,
    max_images=MAX_OOD_IMAGES,
    seed=SEED + 2,
)
eval_images = pd.concat([val_images, ood_images], ignore_index=True)
eval_gt = pd.concat([val_gt, ood_gt], ignore_index=True)

print(f'Train images: {len(train_images)} | train GT boxes: {len(train_gt)}')
print(f'ID eval images: {len(val_images)} | ID GT boxes: {len(val_gt)}')
print(f'OOD eval images: {len(ood_images)} | OOD GT boxes: {len(ood_gt)} | domain={OOD_DOMAIN}')
display(eval_images.groupby('dataset_name').size().rename('images').reset_index())

## Run Detector And Build Image-Class Targets

A row is marked as failed if the detector has any false positive or false negative for that class in that image. GT-only rows represent missed classes. Detection-only rows represent hallucinated classes.

In [ ]:
pred_id = run_torchvision_detector(
    val_images['image_path'].tolist(),
    image_ids=val_images['image_id'].tolist(),
    category_table=category_table,
    model_name=DETECTOR_MODEL,
    score_threshold=DETECTION_SCORE_THRESHOLD,
    batch_size=DETECTOR_BATCH_SIZE,
    detector_name=DETECTOR_MODEL,
    dataset_name='COCO-val',
)
pred_ood = run_torchvision_detector(
    ood_images['image_path'].tolist(),
    image_ids=ood_images['image_id'].tolist(),
    category_table=category_table,
    model_name=DETECTOR_MODEL,
    score_threshold=DETECTION_SCORE_THRESHOLD,
    batch_size=DETECTOR_BATCH_SIZE,
    detector_name=DETECTOR_MODEL,
    dataset_name=f'COCO-O-{OOD_DOMAIN}',
)
predictions = pd.concat([pred_id, pred_ood], ignore_index=True)

tables = evaluate_detection_tables(
    predictions,
    eval_gt,
    iou_threshold=IOU_THRESHOLD,
    backend='auto',
    class_names=class_names,
    detector_name=DETECTOR_MODEL,
)

display(
    tables.image_class_metrics.groupby('dataset_name')
    .agg(rows=('image_id', 'size'), failure_rate=('has_failure', 'mean'), detections=('num_detections', 'sum'), gt=('num_gt', 'sum'))
    .reset_index()
)
display(tables.image_class_metrics.head())

## Fit ViM Reference Scores

ViM needs logits, so the notebook trains a lightweight COCO chip/category head on train chips. The same head is reused for full images and detection chips, which keeps the OOD scoring method consistent across scales.

In [ ]:
_, processor, _ = load_backbone(BACKBONE)
embedder = Embedder(backbone=BACKBONE)
_DL_KW = dict(num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT_WORKERS)

train_paths, train_sizes = image_maps(train_images)
eval_paths, eval_sizes = image_maps(eval_images)

train_chip_source = train_gt.rename(columns={'gt_id': 'detection_id'})
train_chip_anns, _ = detection_chips_from_table(
    train_chip_source,
    image_paths=train_paths,
    image_sizes=train_sizes,
)
train_chip_ds = ChipDataset(train_chip_anns, processor, class_names=category_table.names())
embedder.fit(
    train_chip_ds,
    mode='head',
    epochs=HEAD_EPOCHS,
    batch_size=BATCH_SIZE,
    lr=HEAD_LR,
    save=False,
    **_DL_KW,
)
head = embedder._head  # noqa: SLF001 - notebook research workflow
assert head is not None
W = head.weight.detach().cpu().numpy()
b = head.bias.detach().cpu().numpy()

In [ ]:
train_image_ds = ImagePathDataset(train_images, processor)
eval_image_ds = ImagePathDataset(eval_images, processor)
train_global = embedder.extract(train_image_ds, batch_size=BATCH_SIZE, **_DL_KW)
eval_global = embedder.extract(eval_image_ds, batch_size=BATCH_SIZE, **_DL_KW)
assert train_global.logits is not None and eval_global.logits is not None

global_vim = ViM(W, b, pct_variance=VIM_PCT_VARIANCE).fit(train_global.to_features())
global_scores = pd.DataFrame(
    {
        'image_id': eval_images['image_id'].tolist(),
        'global_ood_score': global_vim.score(eval_global.to_features()),
    }
)

train_chip_res = embedder.extract(train_chip_ds, batch_size=BATCH_SIZE, **_DL_KW)
assert train_chip_res.logits is not None and train_chip_res.labels is not None
chip_vim = ViM(W, b, pct_variance=VIM_PCT_VARIANCE).fit(train_chip_res.to_features())

det_enriched = tables.detections_enriched.copy()
if len(det_enriched):
    chip_anns, ordered_detection_ids = detection_chips_from_table(
        det_enriched,
        image_paths=eval_paths,
        image_sizes=eval_sizes,
    )
    pred_chip_ds = ChipDataset(chip_anns, processor, class_names=category_table.names())
    pred_chip_res = embedder.extract(pred_chip_ds, batch_size=BATCH_SIZE, **_DL_KW)
    assert pred_chip_res.logits is not None
    chip_score_frame = pd.DataFrame(
        {
            'detection_id': ordered_detection_ids,
            'chip_ood_score': chip_vim.score(pred_chip_res.to_features()),
        }
    )
    det_enriched = det_enriched.merge(chip_score_frame, on='detection_id', how='left')
else:
    det_enriched['chip_ood_score'] = []

image_class_features = attach_ood_features(
    tables.image_class_metrics,
    global_scores=global_scores,
    detections_enriched=det_enriched,
    embedding_model_name=BACKBONE,
    ood_detector_name='ViM',
)
image_class_features = add_reference_zscores(
    image_class_features,
    ['global_ood_score', 'chip_ood_mean'],
    reference_domain='COCO-val',
)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

display(image_class_features.groupby('dataset_name')[['global_ood_score', 'chip_ood_mean', 'combined_ood_z']].mean().reset_index())

## Failure Prediction Baselines

These baselines are deliberately simple and use only inference-time features.

- `confidence only`: detector confidence summaries and detection count.
- `OOD only`: global image ViM and detection-chip ViM summaries.
- `confidence + OOD`: combines detector confidence with global and chip OOD features.
- `confidence + OOD + class`: adds class identity as a categorical prior.

Target-derived columns such as `tp`, `fp`, `fn`, `precision`, `recall`, and `has_failure` are evaluation outputs only and must not be used as model inputs.

In [ ]:
CONFIDENCE_FEATURES = [
    'mean_detection_confidence',
    'max_detection_confidence',
    'min_detection_confidence',
    'num_detections',
]
OOD_FEATURES = [
    'global_ood_score',
    'chip_ood_mean',
    'chip_ood_max',
    'chip_ood_p90',
    'chip_ood_std',
    'num_chips',
]
FEATURE_SETS = {
    'failure prior': [],
    'class identity only': ['class_id'],
    'confidence only': CONFIDENCE_FEATURES,
    'OOD only': OOD_FEATURES,
    'confidence + OOD': CONFIDENCE_FEATURES + OOD_FEATURES,
    'confidence + OOD + class': CONFIDENCE_FEATURES + OOD_FEATURES + ['class_id'],
}

TARGET_DERIVED_COLUMNS = {'tp', 'fp', 'fn', 'precision', 'recall', 'f1', 'has_failure'}
MODEL_INPUT_COLUMNS = {col for cols in FEATURE_SETS.values() for col in cols}
assert MODEL_INPUT_COLUMNS.isdisjoint(TARGET_DERIVED_COLUMNS)

metrics, failure_predictions = evaluate_failure_baselines(
    image_class_features,
    target_col='has_failure',
    feature_sets=FEATURE_SETS,
    group_col='image_id',
    test_size=0.3,
    random_state=SEED,
)
print_metrics(metrics)

print('Held-out prediction metrics by domain:')
display(prediction_metrics_by_domain(failure_predictions, image_class_features).sort_values(['dataset_name', 'AUROC'], ascending=[True, False]))

In this COCO/COCO-O cartoon experiment, detector confidence is the strongest baseline. OOD-only features are meaningfully predictive under the cartoon shift, and the key comparison is confidence-only vs confidence + OOD + class.

In [ ]:
best_model = metrics.sort_values('AUROC', ascending=False).iloc[0]['model']
pred_best = failure_predictions[failure_predictions['model'] == best_model].copy()
cal = calibration_bins(pred_best['target'], pred_best['predicted_failure_probability'], n_bins=10)

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot([0, 1], [0, 1], color='0.7', linestyle='--')
ax.scatter(cal['mean_predicted'], cal['observed_rate'], s=np.maximum(cal['count'], 1) * 8)
ax.set_title(f'Calibration: {best_model}')
ax.set_xlabel('Mean predicted failure probability')
ax.set_ylabel('Observed failure rate')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.show()
display(cal)

## High-Confidence Residual Risk

Overall AUROC can hide the value of OOD features because confidence catches many obvious failures. In this experiment, the more relevant question is whether high-OOD rows fail more often among rows with similar confidence.

In [ ]:
high_conf_failures = image_class_features[
    image_class_features['mean_detection_confidence'].ge(HIGH_CONFIDENCE_THRESHOLD)
    & image_class_features['has_failure'].astype(bool)
].sort_values('combined_ood_z', ascending=False)

display(
    high_conf_failures[
        [
            'dataset_name',
            'image_id',
            'class_name',
            'num_gt',
            'num_detections',
            'tp',
            'fp',
            'fn',
            'mean_detection_confidence',
            'global_ood_score_z',
            'chip_ood_mean_z',
            'combined_ood_z',
        ]
    ].head(20)
)

print('Same-confidence failure rates split by OOD terciles, COCO-O only:')
display(
    confidence_ood_failure_table(
        image_class_features[image_class_features['dataset_name'].eq(f'COCO-O-{OOD_DOMAIN}')]
    )
)

print('Same-confidence failure rates split by OOD terciles, all eval rows:')
display(confidence_ood_failure_table(image_class_features))

The high-confidence COCO-O cartoon slice is the main takeaway for this run: OOD helps identify suspicious cases where this detector still appears confident.

## Risk Table And Qualitative Examples

In [ ]:
risk = pred_best.merge(
    image_class_features,
    on=['image_id', 'class_id', 'class_name'],
    how='left',
)
risk = risk.sort_values('predicted_failure_probability', ascending=False)
display(
    risk[
        [
            'dataset_name',
            'image_id',
            'class_name',
            'target',
            'predicted_failure_probability',
            'num_gt',
            'num_detections',
            'tp',
            'fp',
            'fn',
            'mean_detection_confidence',
            'global_ood_score',
            'chip_ood_mean',
            'combined_ood_z',
        ]
    ].head(20)
)

In [ ]:
def show_image_class(row, *, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 5))
    image_id = row['image_id']
    class_id = int(row['class_id'])
    img = Image.open(eval_paths[image_id]).convert('RGB')
    ax.imshow(img)
    ax.axis('off')
    title = f"{row['dataset_name']} | {row['class_name']} | p(fail)={row.get('predicted_failure_probability', np.nan):.2f}"
    ax.set_title(title, fontsize=9)

    gt_rows = tables.ground_truth_enriched[
        (tables.ground_truth_enriched['image_id'] == image_id)
        & (tables.ground_truth_enriched['class_id'] == class_id)
    ]
    det_rows = det_enriched[
        (det_enriched['image_id'] == image_id)
        & (det_enriched['class_id'] == class_id)
    ]
    for _, g in gt_rows.iterrows():
        x1, y1, x2, y2 = g['bbox']
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color='lime', linewidth=2))
    for _, d in det_rows.iterrows():
        x1, y1, x2, y2 = d['bbox']
        color = 'cyan' if d['match_status'] == 'TP' else 'red'
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color=color, linewidth=2, linestyle='--'))
    return ax


gallery = risk[risk['dataset_name'].eq(f'COCO-O-{OOD_DOMAIN}')].head(6)
if len(gallery) < 6:
    gallery = risk.head(6)
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, (_, row) in zip(axes.ravel(), gallery.reset_index(drop=True).iterrows()):
    show_image_class(row, ax=ax)
plt.tight_layout()
plt.show()

print('GT boxes are lime. TP detections are cyan dashed. FP detections are red dashed.')

## Limitations

- This result is specific to this detector, COCO reference data, and COCO-O cartoon split.
- COCO-O cartoon is useful but does not represent all real deployment shifts.
- The failure target is strict.
- Chip OOD depends on detector-predicted boxes, so fully missed objects may not have chip features.
- AUROC measures ranking, not calibration.

In this COCO-O cartoon demo, OOD scores are not a replacement for detector confidence. They provide an additional risk signal for triaging likely failures from this detector under this specific visual shift.